# Web extraction

This notebook shows examples of text extraction from DOCX files with different packages

**Table of contents**<a id='toc0_'></a>    
- 1. [Methods to load from websites](#toc1_)    
  - 1.1. [Load from Async local loader](#toc1_1_)    
    - 1.1.1. [Clean html output from async loader with the Html2Text transformer](#toc1_1_1_)    
    - 1.1.2. [Split output from Html2Text transformer with recursive character text splitter](#toc1_1_2_)    
- 2. [Evaluate loded docs by embedding similarity](#toc2_)    
  - 2.1. [Embedding & Storage](#toc2_1_)    
  - 2.2. [Similarity search](#toc2_2_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=2
	maxLevel=4
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [ ]:
import os
import sys

current_dir = os.getcwd()
kit_dir = os.path.abspath(os.path.join(current_dir, ".."))
repo_dir = os.path.abspath(os.path.join(kit_dir, ".."))

sys.path.append(kit_dir)
sys.path.append(repo_dir)

import pandas as pd
from dotenv import load_dotenv
from langchain.text_splitter import RecursiveCharacterTextSplitter
from tqdm.autonotebook import trange


## 1. <a id='toc1_'></a>[Methods to load from websites](#toc0_)

In [ ]:
urls = [
    "https://en.wikipedia.org/wiki/Unstructured_data",
    "https://unstructured-io.github.io/unstructured/introduction.html",
]

##### Load text splitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
        # Set a small chunk size, just to make splitting evident.
        chunk_size = 500,
        chunk_overlap  = 100,
        length_function = len,
        add_start_index = True,
        separators = ["\n\n\n","\n\n", "\n", "."]
    )

### 1.1. <a id='toc1_1_'></a>[Load from Async local loader](#toc0_)

In [ ]:
from langchain.document_loaders import AsyncHtmlLoader

loader = AsyncHtmlLoader(urls, verify_ssl=False)
docs = loader.load()

for doc in docs:
    print(f'{doc.page_content}\n---')

#### 1.1.1. <a id='toc1_1_1_'></a>[Clean html output from async loader with the Html2Text transformer](#toc0_)

In [ ]:
from langchain.document_transformers import Html2TextTransformer

html2text_transformer = Html2TextTransformer()
docs=html2text_transformer.transform_documents(documents=docs)
for doc in docs:
    print(f'{doc.page_content}\n---')

#### 1.1.2. <a id='toc1_1_2_'></a>[Split output from Html2Text transformer with recursive character text splitter](#toc0_)

In [ ]:
docs = text_splitter.split_documents(docs)
for doc in docs:
    print(f'{doc.page_content}\n---')

## 2. <a id='toc2_'></a>[Evaluate loded docs by embedding similarity](#toc0_)

### 2.1. <a id='toc2_1_'></a>[Embedding & Storage](#toc0_)

In [ ]:
from langchain.embeddings import HuggingFaceInstructEmbeddings
from langchain.vectorstores import FAISS

encode_kwargs = {'normalize_embeddings': True}
embd_model = HuggingFaceInstructEmbeddings( model_name='intfloat/e5-large-v2',
                                            embed_instruction="", # no instructions needed for candidate passages
                                            query_instruction="Represent this sentence for searching relevant passages: ",
                                            encode_kwargs=encode_kwargs)
vectorstore = FAISS.from_documents(documents=docs, embedding=embd_model)

### 2.2. <a id='toc2_2_'></a>[Similarity search](#toc0_)

In [ ]:
query = "how unstructured deal witn ambiguities?"

ans = vectorstore.similarity_search(query)
print("-------Async local Loader + html2text transformer----------\n")
print(ans[0].page_content)

